# 06 — Treinamento YOLOv8: Cliente + Dados Públicos (Combinado)

> **Experimento B** — cliente + público (RWF-2000). YAMLs mesclados sem copiar ficheiros.

**Ordem recomendada:** `01` → `04` → `05_train_public_only` → `02` → `03` → **este notebook**.

Se existir `models/public_only_best.pt`, o treino inicia aí (fine-tune); caso contrário usa `yolov8m.pt`.

**Saída:** `models/combined_best.pt`

In [ ]:
!pip install -q ultralytics matplotlib seaborn pandas PyYAML

In [ ]:
import yaml, json, shutil
import torch
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from ultralytics import YOLO

CLIENT_YAML  = Path("../data/datasets/client_only/dataset.yaml")
PUBLIC_YAML  = Path("../data/datasets/public_only/dataset.yaml")
COMBINED_DIR = Path("../data/datasets/combined")
MODELS_DIR   = Path("../models")
RUNS_DIR     = MODELS_DIR / "runs" / "combined"   # onde o ultralytics salva os runs
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

assert CLIENT_YAML.exists(), "Execute 03_dataset_client_only.ipynb primeiro."
assert PUBLIC_YAML.exists(), "Execute 04_dataset_public.ipynb primeiro."

DEVICE = "0" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")

## 1. Criar dataset combinado via symlinks (sem copiar arquivos)

In [ ]:
# Estratégia: criar um dataset.yaml que aponta para as imagens de AMBOS os datasets
# usando a funcionalidade multi-dataset do Ultralytics
# Syntax: lista de paths no campo 'train', 'val', 'test'

with open(CLIENT_YAML) as f: client_cfg = yaml.safe_load(f)
with open(PUBLIC_YAML) as f: public_cfg = yaml.safe_load(f)

client_path = Path(client_cfg["path"])
public_path = Path(public_cfg["path"])

combined_yaml = dict(
    # Ultralytics suporta lista de caminhos absolutos por split
    train = [
        str(client_path / "images" / "train"),
        str(public_path / "images" / "train"),
    ],
    val = [
        str(client_path / "images" / "val"),
        str(public_path / "images" / "val"),
    ],
    test = [
        str(client_path / "images" / "test"),
        str(public_path / "images" / "test"),
    ],
    nc    = 3,
    names = ["person", "violence", "pre_violence"],
)

COMBINED_DIR.mkdir(parents=True, exist_ok=True)
combined_yaml_path = COMBINED_DIR / "dataset.yaml"
with open(combined_yaml_path, "w") as f:
    yaml.dump(combined_yaml, f, default_flow_style=False)

# Contar frames de cada fonte
n_client = len(list((client_path/"images"/"train").glob("*.jpg")))
n_public = len(list((public_path/"images"/"train").glob("*.jpg")))
print(f"Frames treino — cliente: {n_client}  público: {n_public}  total: {n_client+n_public}")
print(f"Dataset combinado YAML: {combined_yaml_path}")

## 2. Treinamento

In [ ]:
CFG = dict(
    model        = "yolov8m.pt",
    epochs       = 100,
    batch        = 16,
    img_size     = 640,
    lr0          = 0.001,
    lrf          = 0.01,
    momentum     = 0.937,
    weight_decay = 0.0005,
    warmup_epochs= 3,
    patience     = 20,
    optimizer    = "AdamW",
    cls          = 0.7,
    box          = 7.5,
    dfl          = 1.5,
    augment      = True,
    mosaic       = 1.0,
    mixup        = 0.15,
    fliplr       = 0.5,
    degrees      = 10.0,
    save_period  = 10,
)
RUN = f"combined_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

pretrain = MODELS_DIR / "public_only_best.pt"
init_w   = str(pretrain) if pretrain.exists() else CFG["model"]
lr0_use  = CFG["lr0"] * (0.5 if pretrain.exists() else 1.0)
print(f"Inicialização: {init_w}  |  lr0={lr0_use}")
model   = YOLO(init_w)
results = model.train(
    data         = str(combined_yaml_path),
    epochs       = CFG["epochs"],
    imgsz        = CFG["img_size"],
    batch        = CFG["batch"],
    device       = DEVICE,
    lr0          = lr0_use,
    lrf          = CFG["lrf"],
    momentum     = CFG["momentum"],
    weight_decay = CFG["weight_decay"],
    warmup_epochs= CFG["warmup_epochs"],
    patience     = CFG["patience"],
    optimizer    = CFG["optimizer"],
    cls          = CFG["cls"], box=CFG["box"], dfl=CFG["dfl"],
    augment      = CFG["augment"], mosaic=CFG["mosaic"],
    mixup        = CFG["mixup"], fliplr=CFG["fliplr"], degrees=CFG["degrees"],
    project      = str(RUNS_DIR),
    name         = RUN,
    save_period  = CFG["save_period"],
    verbose      = True,
)
print("✓ Treinamento combinado concluído")

In [ ]:
run_dir = RUNS_DIR / RUN
csv_p   = run_dir / "results.csv"

if csv_p.exists():
    df = pd.read_csv(csv_p); df.columns = df.columns.str.strip()
    fig, axes = plt.subplots(1,3,figsize=(15,4))
    for ax,(col,title,color) in zip(axes,[
        ("train/cls_loss","Train Cls Loss","tomato"),
        ("metrics/mAP50(B)","mAP@50","seagreen"),
        ("metrics/recall(B)","Recall","slateblue")]):
        if col in df.columns:
            ax.plot(df["epoch"],df[col],color=color,lw=2)
        ax.set_title(title); ax.grid(alpha=0.3)
    plt.suptitle("Treinamento — Combined",fontsize=13)
    plt.tight_layout(); plt.show()

# Avaliar no teste do cliente (dados sensíveis — avalia separado)
best_w = run_dir / "weights" / "best.pt"
if best_w.exists():
    best_m = YOLO(str(best_w))

    print("\n── Avaliação no teste do CLIENTE:")
    m_client = best_m.val(data=str(CLIENT_YAML), split="test", device=DEVICE)

    print("\n── Avaliação no teste PÚBLICO:")
    m_public = best_m.val(data=str(PUBLIC_YAML), split="test", device=DEVICE)

    result_summary = dict(
        experiment        = "combined",
        weights           = str(best_w),
        # Métricas no cliente
        client_map50      = m_client.box.map50,
        client_map50_95   = m_client.box.map,
        client_precision  = m_client.box.mp,
        client_recall     = m_client.box.mr,
        client_ap50_per_class = {
            "person":      float(m_client.box.ap50[0]) if len(m_client.box.ap50)>0 else None,
            "violence":    float(m_client.box.ap50[1]) if len(m_client.box.ap50)>1 else None,
            "pre_violence":float(m_client.box.ap50[2]) if len(m_client.box.ap50)>2 else None,
        },
        # Métricas no público
        public_map50      = m_public.box.map50,
        public_map50_95   = m_public.box.map,
    )
    with open(MODELS_DIR/"results_combined.json", "w") as f:
        json.dump(result_summary, f, indent=2)

    # Copiar melhor peso para models/ (local esperado pelo nb07 e nb09)
    shutil.copy(str(best_w), str(MODELS_DIR/"combined_best.pt"))
    print(f"\n✓ Pesos: models/combined_best.pt")
    print(f"  mAP@50 cliente : {m_client.box.map50:.4f}")
    print(f"  mAP@50 público : {m_public.box.map50:.4f}")

    # Exportar ONNX ao lado dos pesos
    best_m.export(format="onnx", imgsz=CFG["img_size"], simplify=True, opset=17)
    print("\nPróximo: 07_comparison.ipynb")